In [13]:
from qiskit import *     ####### Fidelity 2026-10-28
import numpy as np
from functools import reduce
from operator import mul

#[j=0]     H2molecule     Fixed-Duratuion  2025-9-2  
n_site=2  
N=n_site 
j_coupling=1
a3=-0.0115
a0=0
a00=-1.04391
a2=-0.42045
J=0.42045 
a1=J
a4=0.179005

step_max=288*3
t_max=36
dt=t_max/step_max

### simulator and Shots
simulator=Aer.get_backend('qasm_simulator')
n_shots=10000 #10^7

##### define quantum circuit
qr=QuantumRegister(14,'qr')
cr=ClassicalRegister(14,'cr')
qc=QuantumCircuit(qr, cr)

### measurement
qc_meas=QuantumCircuit(qr,cr)
qc_meas.measure(qr,cr)

### LIst
#time_list_qiskit=[]
#mag_list_qiskit=[]
#magx_list_qiskit=[]
#magy_list_qiskit=[]

#####  Important : Initial state    |+\rangle|-\rangle

qc.x(qr[13]) 
#qc.x(qr[12])     

#qc.h(qr[13])
#qc.h(qr[12])

#################################   Z:measurement
  
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

############################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                           
        obs=obs+1
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/n_shots
        else:
            mag00=mag00

print('mag00=',mag00)

mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))  
        if j>=n_site-2:                    
            mag=mag+a1*obs*count/n_shots/(1)
        else:
            mag=mag

#mag_list_qiskit.append(mag)
print('mag=',mag)
#################################
#################################
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))  
        if j>=n_site-2:                    
            mag2=mag2+a2*obs*count/n_shots/(1)
        else:
            mag2=mag2
#time_list_qiskit.append(dt*step)
#mag_list_qiskit.append(mag)
print('mag2=',mag2)
#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(n_shots)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)


############################################  measurement:XX-energy
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
qc.h(qr[12])
qc.h(qr[13])

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*count
      
    magx=a4*obs/n_shots
    
print('magx=', magx)

#################################
    
qc_total_zz=qc.compose(qc_meas)
  

job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*count
      
    magzz=a3*obs/n_shots
print('magzz=', magzz)

##################################################
print('EE=',mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)

print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))



mag00= 0.0
mag= -0.42045
mag2= -0.42045
magyy= -0.0236
magx= 0.001288836
magzz= 0.0115
EE= -0.828111164
E0= -1.8720211639999997
F3= 0.010104111955999983


In [14]:
# [j=1]   ### Second-order Suzuki-Trotter formula 
##################################  
import numpy as np
E=1.3939872  ###  set E=E-a00  FIXED DURATION(Important)
ss=0.5*np.pi/(E)
Dt=0.01

qc.h(qr[10])
qc.h(qr[11])

#################################################
qc.s(qr[10])


for step1A in range(100):
 #   qc.p(-1*a0*ss*Dt,qr[10]) 
### JZ0                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[10],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[10],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])

### Z1                                                
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[10],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[10],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
### ZZ
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[10],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[10],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[10],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[10],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[10],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[10],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13])    

### ZZ
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[10],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[10],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    

### Z1                                                
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[10],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[10],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[10],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[10],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
########################################################################### 
for stepBB in range (0):     ### Simple
    qc.s(qr[11])
    
    for step1B in range(100):
        qc.p(-1*a0*ss*Dt,qr[11]) 

### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[11],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[11],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        
### Z1                                                
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])   
        qc.cx(qr[11],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[11],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

#### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
        
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[11],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 

### ZZ  
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[11],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        
        
### Z1                                                
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[11],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[11],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[11],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[11],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
##########################################################################           

#################################################################
    
qc.h(qr[10])
qc.h(qr[11])

######################################### 
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)


zero=0
for bit_string, count in counts.items():
    zero=zero+(1-int(bit_string[2]))*(1-int(bit_string[3]))*count 
print('zero=', zero)

###########################################################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                            
        obs=obs+1*(1-int(bit_string[2]))*(1-int(bit_string[3]))  
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/(zero)
        else:
            mag00=mag00

print('mag00=',mag00)
############################################################
    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))*(1-int(bit_string[2]))*(1-int(bit_string[3]))  
        if j>=n_site-2:                                     
            mag=mag+a1*obs*count/(zero)/(1)
        else:
            mag=mag
#mag_list_qiskit.append(mag)
print('mag=',mag)
#######################################
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))*(1-int(bit_string[2]))*(1-int(bit_string[3]))  
        if j>=n_site-2:                                     
            mag2=mag2+a2*obs*count/(zero)/(1)
        else:
            mag2=mag2
#time_list_qiskit.append(dt*step)
#mag_list_qiskit.append(mag)
print('mag2=',mag2)

#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(zero)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)

############################################  XX-energy
    
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
  #  qc_total=qc+qc_meas  OLD
qc.h(qr[12])
qc.h(qr[13])
    
job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
#print('xx-counts=', counts)
       
magx=0
obs=0
obsy=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magx=a4*obs/(zero)/(1)
    magy=a4*obsy/(zero)/(1)
       
print('magx=', magx)
print('magy=', magy)
##############################################
magxx=0
obsxx=0
for bit_string, count in counts.items():
    for j in range(1):
        obsxx=obsxx+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
                                          
    magxx=a4*obsxx/(zero)/(1)
       
print('magxx=', magxx)
#################################
    
qc_total_zz=qc.compose(qc_meas)
    

job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
#print('xx-counts=', counts)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
       
    magzz=a3*obs/(zero)

print('magzz=', magzz)


print('EE=',mag00+ magzz+mag+mag2+magx)
print('E0=',a00+magzz+mag+mag2+magx)

print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))





zero= 1021
mag00= 0.0
mag= -0.35867967678746326
mag2= -0.35867967678746326
magyy= 0.15768854064642507
magx= 0.023493310479921644
magy= 0.023493310479921644
magxx= 0.023493310479921644
magzz= 0.011454946131243877
EE= -0.682411096963761
E0= -1.726321096963761
F3= 0.09686529227228205


In [15]:
###[j=2]  
from functools import reduce
from operator import mul

E=1.3939872 ###  substitute E=E-a00 Fixed duration

ss=0.5*np.pi/(E)
Dt=0.01

qc.h(qr[8])
qc.h(qr[9])

#################################################
qc.s(qr[8])


for step1A in range(100):
#    qc.p(-1*a0*ss*Dt,qr[8]) 
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[8],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[8],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])

### Z1                                                
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[8],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[8],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
### ZZ
    
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[8],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[8],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 

    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[8],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[8],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 

### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[8],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[8],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    
### ZZ
    
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[8],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[8],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    

### Z1                                                
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[8],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[8],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
    
### JZ                                               
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[8],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[8],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
########################################################################### 
for stepBB in range (0):    ### simple
    qc.s(qr[9])
    

    for step1B in range(100):
        qc.p(-1*a0*ss*Dt,qr[9]) 
### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[9],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[9],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 

### Z1                                              
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[9],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[9],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[9],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[9],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        

### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[9],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[9],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    
    
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[9],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[9],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    

### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[9],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[9],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        

### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[9],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[9],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 


### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[9],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[9],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
##########################################################################           

    
qc.h(qr[8])
qc.h(qr[9])

######################################### 
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

zero=0
for bit_string, count in counts.items():
    zero=zero+(1-int(bit_string[2]))*(1-int(bit_string[3]))*(1-int(bit_string[4]))*(1-int(bit_string[5]))*count 
print('zero=', zero)
   
##########################################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                                 
        obs=obs+1*(1-int(bit_string[2]))*(1-int(bit_string[3]))*(1-int(bit_string[4]))*(1-int(bit_string[5]))
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/(zero)
        else:
            mag00=mag00

print('mag00=',mag00)
######################################################
    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,6)))))     
        if j>=n_site-2:                                     
            mag=mag+a1*obs*count/(zero)/(1)
        else:
            mag=mag

print('mag=',mag)


######################################################
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,6)))))    
        if j>=n_site-2:                                     
            mag2=mag2+a2*obs*count/(zero)/(1)
        else:
            mag2=mag2

print('mag2=',mag2)

#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,6)))))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(zero)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)


############################################  XX-energy
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
  
qc.h(qr[12])
qc.h(qr[13])
    

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)

       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,6)))))*count #*(1-int(bit_string[2]))*(1-int(bit_string[3]))*(1-int(bit_string[4]))*(1-int(bit_string[5]))*count
       
        magx=a4*obs/(zero)
#magx_list_qiskit.append(magx)
print('magx=', magx)

#################################
    

qc_total_zz=qc.compose(qc_meas)

job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,6)))))*count 
       
    magzz=a3*obs/(zero)
print('magzz=', magzz)

##########################################
print('EE=', mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)

print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))














zero= 179
mag00= 0.0
mag= 0.07281536312849163
mag2= 0.07281536312849163
magyy= -0.30726256983240224
magx= -0.062001731843575415
magzz= 0.01194972067039106
EE= 0.09557871508379892
E0= -0.948331284916201
F3= 0.5604483763687151


In [16]:
#[j=3]

#E=0.870522  ###  substitute E=E-a00  Fixed duration
ss=0.5*np.pi/(E)
Dt=0.01

qc.h(qr[6])
qc.h(qr[7])

#################################################
qc.s(qr[6])


for step1A in range(100):
#    qc.p(-1*a0*ss*Dt,qr[6]) 
### JZ                                              
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[6],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[6],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])
    
### Z1                                               
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[6],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[6],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
### ZZ

    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[6],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[6],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12])
    
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[6],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[6],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    
    
#### YY nashi

### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[6],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[6],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    

### ZZ
    
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[6],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[6],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    
### Z1                                           
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[6],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[6],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
    
### JZ                                               
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[6],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[6],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
#################################################
for stepBB in range (0): ### Simple
    qc.s(qr[7])
     

    for step1B in range(100):
        qc.p(-1*a0*ss*Dt,qr[7])
### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[7],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[7],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 

### Z1                                              
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[7],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[7],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[7],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[7],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[7],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[7],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 

### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[7],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[7],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    

### ZZ
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[7],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[7],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12])
        
### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[7],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[7],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[7],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[7],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 

#################################################################
    
qc.h(qr[6])
qc.h(qr[7])

######################################### 2023-1-2
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

zero=0
for bit_string, count in counts.items():
    zero=zero+((reduce(mul,((1-int(bit_string[i])) for i in range(2,8))))) *count 
print('zero=', zero)

###########################################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                                 
        obs=obs+1*((reduce(mul,((1-int(bit_string[i])) for i in range(2,8)))))
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/(zero)
        else:
            mag00=mag00

print('mag00=',mag00)
#############################################
    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,8)))))    
        if j>=n_site-2:                                     
            mag=mag+a1*obs*count/(zero)/(1)
        else:
            mag=mag
#time_list_qiskit.append(dt*step)
#mag_list_qiskit.append(mag)
print('mag=',mag)

#############################################
    
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,8)))))    
        if j>=n_site-2:                                     
            mag2=mag2+a2*obs*count/(zero)/(1)
        else:
            mag2=mag2
print('mag2=',mag2)
#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,8)))))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(zero)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)


############################################  
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
qc.h(qr[12])
qc.h(qr[13])
    

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
#print('xx-counts=', counts)
       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,8)))))*count 
    magx=a4*obs/(zero)

print('magx=', magx)

#################################
    
qc_total_zz=qc.compose(qc_meas)
  
    

job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,8)))))*count 
    magzz=a3*obs/(zero)
print('magzz=', magzz)

########################################
print('EE=', mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)

print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))





zero= 79
mag00= 0.0
mag= 0.3991613924050633
mag2= 0.3991613924050633
magyy= -0.43037974683544306
magx= -0.06571069620253164
magzz= 0.014848101265822785
EE= 0.7474601898734178
E0= -0.2964498101265821
F3= 0.9955558262025317


In [17]:
#[j=4]  

#E=0.870522 # substutute E=E-a00
ss=0.5*np.pi/(E)
Dt=0.01

qc.h(qr[4])
qc.h(qr[5])

#################################################
qc.s(qr[4])

for step1A in range(100):
    qc.p(-1*a0*ss*Dt,qr[4]) 
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[4],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[4],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])
    
### Z1                                               
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[4],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[4],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
### ZZ

    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[4],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[4],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12])
    
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[4],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[4],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 

### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[4],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[4],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    

### ZZ
    
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[4],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[4],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    
### Z1                                              
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[4],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[4],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
       
### JZ                                               
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[4],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[4],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
###################################################
for stepBB in range (0): ### Simple
    qc.s(qr[5])
     

    for step1B in range(100):
        qc.p(-1*a0*ss*Dt,qr[5])
### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[5],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[5],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 

### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[5],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[5],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    

### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[5],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[5],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    

### ZZ
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12])
        
### Z1                                              
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[5],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[5],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[5],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[5],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
######################################################
    
qc.h(qr[4])
qc.h(qr[5])

######################################### 
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

zero=0
for bit_string, count in counts.items():
    zero=zero+((reduce(mul,((1-int(bit_string[i])) for i in range(2,10))))) *count 
print('zero=', zero)

###########################################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                           
        obs=obs+1*((reduce(mul,((1-int(bit_string[i])) for i in range(2,10)))))
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/(zero)
        else:
            mag00=mag00

print('mag00=',mag00)
#############################################
    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,10)))))    
        if j>=n_site-2:                                     
            mag=mag+a1*obs*count/(zero)/(1)
        else:
            mag=mag
#mag_list_qiskit.append(mag)
print('mag=',mag)

#############################################
    
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,10)))))    
        if j>=n_site-2:                                   
            mag2=mag2+a2*obs*count/(zero)/(1)
        else:
            mag2=mag2
print('mag2=',mag2)
#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,10)))))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(zero)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)


############################################  XX-energy
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
  
qc.h(qr[12])
qc.h(qr[13])
    

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,10)))))*count #*(1-int(bit_string[2]))*(1-int(bit_string[3]))*(1-int(bit_string[4]))*(1-int(bit_string[5]))*count
    magx=a4*obs/(zero)
print('magx=', magx)

#################################
    
qc_total_zz=qc.compose(qc_meas)
 
job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,10)))))*count 
    magzz=a3*obs/(zero)
print('magzz=', magzz)

########################################
print('EE=', mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)

print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))




zero= 82
mag00= 0.0
mag= 0.42045
mag2= 0.42045
magyy= 0.08536585365853659
magx= 0.015280914634146341
magzz= 0.010939024390243903
EE= 0.8671199390243902
E0= -0.1767900609756097
F3= 0.9857339953658537


In [18]:
#[j=5]  

#E=0.870522 # substitute E=E-a00   Fixed duration
ss=0.5*np.pi/(E)
Dt=0.01

qc.h(qr[2])
qc.h(qr[3])

#################################################
qc.s(qr[2])


for step1A in range(100):
 #   qc.p(-1*a0*ss*Dt,qr[2]) 
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[2],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[2],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])
    
### Z1                                               
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[2],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[2],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
### ZZ

    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[2],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[2],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12])
    
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[2],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[2],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[2],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[2],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    

### ZZ
    
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[2],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[2],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    
### Z1                                               
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[2],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[2],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
    
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[2],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[2],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
########################################################################### 
for stepBB in range (0): ### Simple
    qc.s(qr[3])
     

    for step1B in range(100):
        qc.p(-1*a0*ss*Dt,qr[3])
### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[3],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[3],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 

### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[3],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[3],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[3],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[3],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[3],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[3],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[3],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[3],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    

### ZZ
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[3],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[3],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12])
        
### Z1                                             
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[3],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[3],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[3],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[3],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
#################################################
    
qc.h(qr[2])
qc.h(qr[3])

#####################################
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

zero=0
for bit_string, count in counts.items():
    zero=zero+((reduce(mul,((1-int(bit_string[i])) for i in range(2,12))))) *count 
print('zero=', zero)

###########################################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                           
        obs=obs+1*((reduce(mul,((1-int(bit_string[i])) for i in range(2,12)))))
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/(zero)
        else:
            mag00=mag00

print('mag00=',mag00)
#############################################
    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,12)))))   
        if j>=n_site-2:                                     
            mag=mag+a1*obs*count/(zero)/(1)
        else:
            mag=mag
print('mag=',mag)

#############################################
    
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,12)))))    
        if j>=n_site-2:                                     
            mag2=mag2+a2*obs*count/(zero)/(1)
        else:
            mag2=mag2
print('mag2=',mag2)
#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,12)))))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(zero)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)


############################################  XX-energy
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
qc.h(qr[12])
qc.h(qr[13])
    

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,12)))))*count 
    magx=a4*obs/(zero)
print('magx=', magx)

#################################
    
qc_total_zz=qc.compose(qc_meas)
    
job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,12)))))*count 
    magzz=a3*obs/(zero)
print('magzz=', magzz)

####################################
print('EE=', mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)



print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))



zero= 75
mag00= 0.0
mag= 0.42045
mag2= 0.42045
magyy= 0.14666666666666667
magx= 0.042961200000000005
magzz= 0.009046666666666666
EE= 0.8929078666666667
E0= -0.15100213333333326
F3= 0.9558355678666666


In [19]:
#[j=6]  

#E=0.870522 # substitute E=E-a00   Fixed duration
ss=0.5*np.pi/(E)
Dt=0.01

qc.h(qr[0])
qc.h(qr[1])

#################################################
qc.s(qr[0])


for step1A in range(100):
 #   qc.p(-1*a0*ss*Dt,qr[0]) 
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[0],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[0],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])
    
### Z1                                               
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[0],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[0],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
### ZZ

    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[0],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[0],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12])
    
    
### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[0],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[0],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    

### XX
    qc.cx(qr[12],qr[13])    
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[0],qr[12])
    qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
    qc.cz(qr[0],qr[12])
    qc.rx((0.25*ss)*a4*Dt,qr[12]) 
    qc.cx(qr[12],qr[13]) 
    

### ZZ
    
    qc.cx(qr[13],qr[12])    
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[0],qr[12])
    qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[0],qr[12])
    qc.rz((0.25*ss)*a3*Dt,qr[12]) 
    qc.cx(qr[13],qr[12]) 
    
### Z1                                               
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
    qc.cx(qr[0],qr[12])
    qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
    qc.cx(qr[0],qr[12])
    qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
     
    
### JZ                                                
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13])  
    qc.cx(qr[0],qr[13])
    qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
    qc.cx(qr[0],qr[13])
    qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
#############################################
for stepBB in range (0): ### Simple
    qc.s(qr[1])
     

    for step1B in range(100):
#        qc.p(-1*a0*ss*Dt,qr[1])
### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[1],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[1],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 

### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12])  
        qc.cx(qr[1],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### ZZ
        
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12]) 
        
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[1],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[1],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    
### XX
        qc.cx(qr[12],qr[13])    
        qc.rx((0.25*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[1],qr[12])
        qc.rx((-0.5*ss)*a4*Dt,qr[12]) 
        qc.cz(qr[1],qr[12])
        qc.rx((0.25*ss)*a4*Dt,qr[12])
        qc.cx(qr[12],qr[13]) 
    

### ZZ
        qc.cx(qr[13],qr[12])    
        qc.rz((0.25*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((-0.5*ss)*a3*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((0.25*ss)*a3*Dt,qr[12])
        qc.cx(qr[13],qr[12])
        
### Z1                                               
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((-0.5*a2*ss)*(step_max/step_max)*Dt,qr[12]) 
        qc.cx(qr[1],qr[12])
        qc.rz((0.25*a2*ss)*(step_max/step_max)*Dt,qr[12]) 

### JZ  
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[1],qr[13])
        qc.rz((-0.5*J*ss)*(step_max/step_max)*Dt,qr[13]) 
        qc.cx(qr[1],qr[13])
        qc.rz((0.25*J*ss)*(step_max/step_max)*Dt,qr[13]) 
###################################################
    
qc.h(qr[0])
qc.h(qr[1])

######################################
qc_total=qc.compose(qc_meas)

job=execute(qc_total, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total)

zero=0
for bit_string, count in counts.items():
    zero=zero+((reduce(mul,((1-int(bit_string[i])) for i in range(2,14))))) *count 
print('zero=', zero)
   
###########################################
mag00=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                       
        obs=obs+1*((reduce(mul,((1-int(bit_string[i])) for i in range(2,14)))))
        if j>=n_site-2:                                     
            mag00=mag00+a0*obs*count/(zero)
        else:
            mag00=mag00

print('mag00=',mag00)
#######################################
    
mag=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,14)))))    
        if j>=n_site-2:                                     
            mag=mag+a1*obs*count/(zero)/(1)
        else:
            mag=mag
print('mag=',mag)

#############################################
    
mag2=0
for bit_string, count in counts.items():
    obs=0
    for j in range(n_site-1):                               
        obs=obs+(-1)**(int(bit_string[j+1]))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,14)))))    
        if j>=n_site-2:                                    
            mag2=mag2+a2*obs*count/(zero)/(1)
        else:
            mag2=mag2
print('mag2=',mag2)
#############################################YY-energy
qc.rx(0.5*np.pi, qr[12])
qc.rx(0.5*np.pi, qr[13])
qc_total_yy=qc.compose(qc_meas)
qc.rx(-0.5*np.pi, qr[12])
qc.rx(-0.5*np.pi, qr[13])

job=execute(qc_total_yy, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_yy)

magyy=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[j]))+(int(bit_string[j+1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,14)))))*count
#        obsy=obsy+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*(1-int(bit_string[2]))*(1-int(bit_string[3]))*count
        
    magyy=obs/(zero)/(1)
#    magy=a4*obsy/(zero)/(1)
       
print('magyy=', magyy)
#print('magy=', magy)


############################################  XX-energy
qc.h(qr[12])
qc.h(qr[13])
qc_total_xx=qc.compose(qc_meas)
qc.h(qr[12])
qc.h(qr[13])
    

job=execute(qc_total_xx, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_xx)
#print('xx-counts=', counts)
       
magx=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,14)))))*count 
    magx=a4*obs/(zero)
print('magx=', magx)

#################################
    
qc_total_zz=qc.compose(qc_meas)
  
job=execute(qc_total_zz, simulator, shots=n_shots)
result=job.result()
counts=result.get_counts(qc_total_zz)
       
magzz=0
obs=0
for bit_string, count in counts.items():
    for j in range(1):
        obs=obs+((-1)**((int(bit_string[0]))+(int(bit_string[1]))))*((reduce(mul,((1-int(bit_string[i])) for i in range(2,14)))))*count 
    magzz=a3*obs/(zero)
print('magzz=', magzz)

###################################
print('EE=', mag00+magzz+mag+mag2+magx)
print('E0=', a00+magzz+mag+mag2+magx)


print('F3=',0.25-0.25*magzz/a3+0.24452112*(mag/a1-mag2/a2)+0.05205171*(magx/a4+magyy))






zero= 69
mag00= 0.0
mag= 0.39607608695652174
mag2= 0.39607608695652174
magyy= 0.07246376811594203
magx= 0.07523398550724637
magzz= 0.0115
EE= 0.8788861594202899
E0= -0.16502384057970998
F3= 0.9863406339130435
